In [32]:
import json
import os
import pandas as pd
from pathlib import Path
pd.options.mode.chained_assignment = None #don't show warning flags
%matplotlib inline

from IPython.display import display, HTML
display(HTML(
    '<style>'
        '#notebook { padding-top:0px !important; } ' 
        '.container { width:97% !important; } '
        '.end_space { min-height:0px !important; } '
    '</style>'
))

pd.set_option('display.max_columns', None)

## Functions

In [47]:
### Generate Raw Data ###
def find_modern_files(data_dir, format_filter):
    modern_files = [
        json_file
        for json_file in data_dir.rglob("*.json")
        if format_filter in json_file.name.lower()
    ]
    return modern_files

def parse_tournament(filepath): # parse a single file to test logic
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    tournament = data.get("Tournament", {}) or {}
    decks = data.get("Decks", []) or []
    standings = data.get("Standings", []) or []
    standings_lookup = {s["Player"]: s for s in standings if "Player" in s}
    records = []
    for deck in decks:
        player = deck.get("Player", "unknown")
        standing = standings_lookup.get(player, {})

        # parse mainboard and sideboard together, default to empty list if None
        for is_mainboard, section in [(True, "Mainboard"), (False, "Sideboard")]:
            for card in (deck.get(section) or []):
                records.append({
                    "card_name": card.get("CardName", "").strip(),
                    "card_count": card.get("Count", 0),
                    "is_mainboard": is_mainboard,
                    "player": player,
                    "player_wins": standing.get("Wins", 0),
                    "player_losses": standing.get("Losses", 0),
                    "tournament_name": tournament.get("Name", "unknown"),
                    "tournament_date": tournament.get("Date", "unknown"),
                    "source_file": filepath.name,
                })
    return records
    
def parse_tournament_data(modern_files):
    all_records = []
    errors = []

    for i, filepath in enumerate(modern_files, 1):
        try:
            records = parse_tournament(filepath)
            all_records.extend(records)
        except Exception as e:
            errors.append({"file": filepath.name, "error": str(e)})

        if i % 100 == 0 or i == len(modern_files):
            print(f"  processed {i}/{len(modern_files)} files — {len(all_records):,} records so far")

    print(f"\ndone — {len(errors)} files failed to parse")
    if errors:
        for err in errors:
            print(f"  {err['file']}: {err['error']}")
    return all_records

def explore_tournament_results(df):
    print(f"shape: {df.shape}")
    print(f"\ncolumns: {df.columns.tolist()}")
    print(f"\ndata types:\n{df.dtypes}")
    print(f"\nmissing values:\n{df.isnull().sum()}")
    print(f"\nunique cards: {df['card_name'].nunique():,}")
    print(f"unique players: {df['player'].nunique():,}")
    print(f"unique tournaments: {df['source_file'].nunique():,}")
    
### Add additional calculation fields ###
def explore_win_records(tournament_results):
    tournament_results["record"] = tournament_results["player_wins"].astype(str) + "-" + tournament_results["player_losses"].astype(str)
    record_counts = tournament_results["record"].value_counts().sort_index()
    pd.set_option("display.max_rows", None)
    print(record_counts)
    
def add_tournament_size(tournament_results):
    tournament_size = tournament_results.groupby("source_file")["player"].nunique().reset_index()
    tournament_size.columns = ["source_file", "tournament_size"]

    # merge back onto main results
    tournament_results = tournament_results.merge(tournament_size, on="source_file", how="left")

    # preview size distribution
    print("tournament size distribution:")
    print(tournament_results.groupby("source_file")["tournament_size"].first().describe())
    return tournament_results
    
def add_tournament_size_bucket(tournament_results):
    def assign_bucket(size):
        if size < 30:
            return "small"
        elif size < 60:
            return "medium"
        elif size < 150:
            return "large"
        else:
            return "major"

    tournament_results["tournament_size_bucket"] = tournament_results["tournament_size"].apply(assign_bucket)
    print("size bucket distribution:")
    print(tournament_results.groupby("tournament_size_bucket")["source_file"].nunique())
    return tournament_results

def filter_strong_results(tournament_results, total_match_cutoff, win_rate_cutoff):
    #this is after tournament size bucket because we need player count before reducing to strong win rates
    total_matches = tournament_results["player_wins"] + tournament_results["player_losses"]
    tournament_results["win_rate"] = tournament_results["player_wins"] / total_matches

    filtered = tournament_results[
        (total_matches >= total_match_cutoff) &
        (tournament_results["win_rate"] >= win_rate_cutoff)
    ]
    print(f"records before filter: {len(tournament_results):,}")
    print(f"records after filter: {len(filtered):,}")
    print(f"records removed: {len(tournament_results) - len(filtered):,}")
    return filtered

### Write file ###
def save_tournament_results(tournament_results, filepath):
    filepath.parent.mkdir(parents=True, exist_ok=True)
    tournament_results.to_csv(filepath, index=False)
    print(f"saved {len(tournament_results):,} records to {filepath}")
    
def output_raw_file():
    raw_output_file = output_dir_raw / "raw_card_records_2016_2018.csv"
    save_tournament_results(raw_df, raw_output_file)

def output_filtered_file():
    filtered_output_file = output_dir_processed / "filtered_card_records_2016_2018.csv"
    save_tournament_results(filtered_df, filtered_output_file)

## Code

### Universal Variables


In [50]:
data_dir = Path("C:/Users/david/Documents/GitHub/ai_projects/modern_cube/data/raw/modern_tournament_data_overall")
output_dir_raw = Path("../data/raw")
output_dir_processed = Path("../data/processed")

format_filter = "modern"

total_match_cutoff = 3 #ommit smaller tournaments
win_rate_cutoff = 0.75 # we only want to include decks that win 75% of the time

### Executed Code

In [35]:
modern_files = find_modern_files(data_dir, format_filter)

In [36]:
all_records = parse_tournament_data(modern_files)
raw_df = pd.DataFrame(all_records)
raw_df.head()

  processed 100/1186 files — 75,972 records so far
  processed 200/1186 files — 170,681 records so far
  processed 300/1186 files — 257,490 records so far
  processed 400/1186 files — 462,146 records so far
  processed 500/1186 files — 651,168 records so far
  processed 600/1186 files — 832,344 records so far
  processed 700/1186 files — 955,055 records so far
  processed 800/1186 files — 1,110,299 records so far
  processed 900/1186 files — 1,255,473 records so far
  processed 1000/1186 files — 1,424,440 records so far
  processed 1100/1186 files — 1,547,298 records so far
  processed 1186/1186 files — 1,661,459 records so far

done — 0 files failed to parse


,card_name,card_count,is_mainboard,player,player_wins,player_losses,tournament_name,tournament_date,source_file
0,Ancient Stirrings,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json
1,Chromatic Sphere,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json
2,Chromatic Star,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json
3,"Emrakul, the Aeons Torn",1,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json
4,Expedition Map,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json


In [37]:
raw_df = add_tournament_size(raw_df)

tournament size distribution:
count    1186.000000
mean       46.495784
std        43.935688
min         4.000000
25%        28.000000
50%        36.500000
75%        47.000000
max       418.000000
Name: tournament_size, dtype: float64


In [45]:
output_raw_file()

saved 1,661,459 records to ..\data\processed\raw_card_records_2016_2018.csv


In [38]:
explore_tournament_results(raw_df)

shape: (1661459, 10)

columns: ['card_name', 'card_count', 'is_mainboard', 'player', 'player_wins', 'player_losses', 'tournament_name', 'tournament_date', 'source_file', 'tournament_size']

data types:
card_name          object
card_count          int64
is_mainboard         bool
player             object
player_wins         int64
player_losses       int64
tournament_name    object
tournament_date    object
source_file        object
tournament_size     int64
dtype: object

missing values:
card_name          0
card_count         0
is_mainboard       0
player             0
player_wins        0
player_losses      0
tournament_name    0
tournament_date    0
source_file        0
tournament_size    0
dtype: int64

unique cards: 3,711
unique players: 12,007
unique tournaments: 1,186


In [39]:
explore_win_records(raw_df)

record
0-0     858847
0-1      13245
0-2      91235
0-3      38421
0-4       5653
0-5        772
0-6        517
1-0        145
1-1       2826
1-2      71629
1-3      63500
1-4      10579
1-5       2147
1-6        512
1-7        127
10-0       324
10-1       495
10-2       346
10-3        27
11-0        41
11-1       338
11-2        62
12-0        65
13-0        29
2-0         57
2-1       3648
2-2      50676
2-3      59106
2-4      17275
2-5       3469
2-6        529
3-0        255
3-1      38275
3-2      50772
3-3      23043
3-4      28838
3-5       2282
3-6        139
3-7         30
4-0      10302
4-1      26669
4-2       1657
4-3      50850
4-4      13588
4-5       1548
4-6        150
5-0       4817
5-1         56
5-2      30252
5-3      19373
5-4       7271
5-5        319
6-1        273
6-2      13675
6-3      14026
6-4       1194
7-0         25
7-1        734
7-2      12636
7-3       2737
8-1        395
8-2       4879
8-3        405
9-1       1680
9-2       1484
9-3        188
Nam

In [40]:
filtered_df = filter_strong_results(raw_df, total_match_cutoff, win_rate_cutoff)

records before filter: 1,661,459
records after filter: 118,070
records removed: 1,543,389


In [41]:
filtered_df.head()

,card_name,card_count,is_mainboard,player,player_wins,player_losses,tournament_name,tournament_date,source_file,tournament_size,record,win_rate
0,Ancient Stirrings,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0
1,Chromatic Sphere,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0
2,Chromatic Star,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0
3,"Emrakul, the Aeons Torn",1,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0
4,Expedition Map,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0


In [42]:
filtered_df = add_tournament_size_bucket(filtered_df)

size bucket distribution:
tournament_size_bucket
large     106
major      23
medium    142
small     207
Name: source_file, dtype: int64


In [43]:
filtered_df.head()

,card_name,card_count,is_mainboard,player,player_wins,player_losses,tournament_name,tournament_date,source_file,tournament_size,record,win_rate,tournament_size_bucket
0,Ancient Stirrings,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small
1,Chromatic Sphere,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small
2,Chromatic Star,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small
3,"Emrakul, the Aeons Torn",1,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small
4,Expedition Map,4,True,BoozeMongoose,4,0,Modern Daily Swiss,2016-01-01T00:00:00Z,modern-daily-swiss-2016-01-019177191.json,24,4-0,1.0,small


In [48]:
output_filtered_file()

saved 118,070 records to ..\data\processed\filtered_card_records_2016_2018.csv
